In [1]:
import numpy as np
import glob
import os
import re
import sys

# This finds the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# --- THIS IS THE LINE YOU ARE MISSING ---
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
# --- ADD THAT LINE! ---

print(f"Project root added to path: {PROJECT_ROOT}")


Project root added to path: /Users/minglu/Documents/Uni/bistable_model_simulation


In [34]:

# If the only difference is P/tau vs kappa, then:
# Method 2 recovered ell ~ ell_method1(kappa_eff) where kappa_eff = P/tau
# Let's check: how much does ell change when kappa shifts by -kappa^2*tau/2?

D = 1500.0
sigma = 0.1
kappa_1p = 716.88


for tau in [1e-6, 2e-7]:

    kappa_eff_1p = (1 - np.exp(-kappa_1p*tau))/tau
    # Method 1 formula: ell = 4pi*(D)*(sigma - tanh(alpha*sigma)/alpha)
    # with alpha = sqrt(kappa/D), D here is D_rel
    D_rel = 2*D  # D_X2 + D_A
    
    def ell_formula(kappa, D_rel, sigma):
        alpha = np.sqrt(kappa/D_rel)
        return 2*np.pi*D_rel*(sigma - np.tanh(alpha*sigma)/alpha)
    
    ell_goal_1 = ell_formula(kappa_1p, D_rel, sigma)
    ell_eff_1 = ell_formula(kappa_eff_1p, D_rel, sigma)

    
    print(f'tau={tau:.0e}:')
    print(f'  kappa_eff_1p/kappa_1p = {kappa_eff_1p/kappa_1p:.6f}')
    print(f'  ell_1p: goal={ell_goal_1:.4f}, eff={ell_eff_1:.4f}')
    print()

tau=1e-06:
  kappa_eff_1p/kappa_1p = 0.999642
  ell_1p: goal=1.5000, eff=1.4995

tau=2e-07:
  kappa_eff_1p/kappa_1p = 0.999928
  ell_1p: goal=1.5000, eff=1.4999



In [2]:
np.sqrt(2*1500*2*1e-7)

np.float64(0.02449489742783178)

In [3]:
from simulation.solvers.spatial_process import simul_initialize, simul_run
from simulation.solvers.rate_conversions import calculate_kappas


from pathlib import Path
import os

In [4]:
""" Main execution block containing all physics parameters. """
###### ================================== 1. parameter setting =====================================
L = 2. # cubic box length

diff_scale = 1500. 
DA = 1. 
DB = 1. 
DX = 1.  
DX2 = 1. 

##### There are 6 reactions but only 4 sigma values
##### because the reactions B <-> X involve no sigma value
sigmas = np.array((1., 1., 1., 1.)) * 0.1 # sigma_r1f, sigma_r1b, sigma_r2f, sigma_r2b

box_shape = np.array((L, L, L,))

##### the Part to change freely for the corresponding simulation
# Schloegl's model reaction rates
k = np.array((0.15, 0.025, 5.75, 25.))
print("Reaction rates for bistable schloegl's model: ",k)
# full model reaction rates
ls = np.array((1.5, 1500., 150., 25., 5.75, 25.))
# ls = np.array((3., 1500., 75., 12.5, 5.75, 25.))
print("Reaction rates for bistable full model: ",ls)
    

Reaction rates for bistable schloegl's model:  [ 0.15   0.025  5.75  25.   ]
Reaction rates for bistable full model:  [   1.5  1500.    150.     25.      5.75   25.  ]


In [16]:
def coeff(k1,k2,k3,k4,a,b,n,vol):
    variable_lambda = a*k1*n*(n-1)/vol+b*k3*vol
    variable_mu = n*k4+k2*n*(n-1)*(n-2)/vol**2
    return variable_lambda, variable_mu

In [23]:
vec_n = np.arange(400)
print(vec_n)

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 24

In [24]:
k1,k2,k3,k4=k
vol=8.
a,b=10., 20.

In [25]:
v_lambda, v_mu = coeff(k1,k2,k3,k4,a,b,vec_n,vol)

In [26]:
np.prod(v_lambda[:-1]/v_mu[1:])

np.float64(21724392504285.184)

In [28]:
p_sum=1
for i in range(len(v_lambda)):
    p_sum = p_sum + np.prod((v_lambda[:-1]/v_mu[1:])[:i])
p0ss=1/p_sum

In [30]:
print(p0ss)# this does not work, it is still a numerical solution, perhaps just call the distribution from the generator matrix theoretical??

2.334879365588944e-19


In [5]:
print("The difference of propensity for second channel on macroscopic level:")
print(f"Low state: {ls[2]*10-ls[3]*60/8}")
print(f"High state: {ls[2]*10-ls[3]*280/8}")
print("The difference of propensity for third channel on macroscopic level:")
print(f"Low state: {ls[4]*20-ls[5]*60/8}")
print(f"High state: {ls[4]*20-ls[5]*280/8}")

The difference of propensity for second channel on macroscopic level:
Low state: 1312.5
High state: 625.0
The difference of propensity for third channel on macroscopic level:
Low state: -72.5
High state: -760.0


In [6]:
range = [0.2, 0.5, 1.0, 2.0, 5.0, 10.0]

In [7]:
for i in range:
    print(f"Current diffusion coeff is: {diff_scale/i}.")
    diffusions = np.array((DX, DX2, DA, DB)) * diff_scale / i
    print(f"Diffusions are : {diffusions}.")
    kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)
    print("The difference of propensity for second channel on microscopic level:")
    print(f"Low state: {kappas[2]*10-kappas[3]*60/8}")
    print(f"High state: {kappas[2]*10-kappas[3]*280/8}")

Current diffusion coeff is: 7500.0.
Diffusions are : [7500. 7500. 7500. 7500.].
✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1633e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.6213e+04
κ₂⁻ = 6.0355e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.
The difference of propensity for second channel on microscopic level:
Low state: 316865.97637919366
High state: 150888.56018056843
Current diffusion coeff is: 3000.0.
Diffusions are : [3000. 3000. 3000. 3000.].
✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1654e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.6835e+04
κ₂⁻ = 6.1392e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.
The difference of propensity for second channel on microscopic level:
Low state: 322310.2647123682
High state: 153481.078434461
Current diffusion

In [8]:
diffusions = np.array((DX, DX2, DA, DB)) * 3000 # 1500
kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1654e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.6835e+04
κ₂⁻ = 6.1392e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.


In [9]:
600/0.01

60000.0

In [10]:
ls = np.array((1.5, 1500., 150., 25., 5.75, 25.))
diffusions = np.array((1500, 100, 100, 100)) # diffusions = np.array((DX, DX2, DA, DB)) # diffusions = np.array((1500, 100, 100, 100))
#def calculate_kappas(ls, DA, DX, DX2, sigma, verbose=True):
kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.1688e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 1.2509e+05
κ₂⁻ = 2.0848e+04
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.


### try this

In [11]:
ls = np.array((1.5, 1500., 130., 25., 5.75, 25.))
# diffusions = np.array((1500, 100, 100, 100)) # diffusions = np.array((DX, DX2, DA, DB)) # diffusions = np.array((1500, 100, 100, 100))
diffusions = np.array((15, 1500, 15, 15)) #np.array((150, 1500, 150, 1500))
#def calculate_kappas(ls, DA, DX, DX2, sigma, verbose=True):
print(diffusions)
kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

[  15 1500   15   15]
✅ R1 Solver converged.
✅ R2 Solver converged and passed physics checks.

--- Final Intrinsic Rates (Kappa) ---
κ₁⁺ = 7.9171e+02
κ₁⁻ = 1.5000e+03
κ₂⁺ = 3.4391e+04
κ₂⁻ = 6.6136e+03
κ₃⁺ = 5.7500e+00
κ₃⁻ = 2.5000e+01
The estimation is calculated based on the Eq. (32) in Erban's paper.


In [12]:
450/0.01

45000.0

## calculate the dimensionaless kappa on the paper

In [13]:
def calculate_k_from_l(l):
    keq = l[0]/l[1]
    k = np.array((keq*l[2], keq*l[3], l[4], l[5]))
    return k

def get_reaction_volume(sigma):
    """Calculates the volume of the reaction sphere."""
    return (4.0/3.0) * np.pi * (sigma**3)


# --- Formula definitions ---

def l2_formula_notcoupled(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 4 * np.pi * (D + D) * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs


In [14]:
print("The difference of propensity for third channel on microscopic level:")
print(f"Low state: {kappas[4]*20-kappas[5]*60/8}")
print(f"High state: {kappas[4]*20-kappas[5]*280/8}")

The difference of propensity for third channel on microscopic level:
Low state: -72.5
High state: -760.0


In [15]:
def l1_plus_formula(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 4 * np.pi * D * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs

def calculate_l2_rates(kappa_2_plus, kappa_2_minus, DA, DX, DX2, sigma_3):
    if kappa_2_plus <= 0 or kappa_2_minus <= 0: 
        return np.inf, np.inf
    
    alpha_sq = kappa_2_plus / (DX2 + DA) + kappa_2_minus / (DX2 + DX)
    alpha = np.sqrt(alpha_sq)
    common_factor = 4 * np.pi * (1 / alpha_sq) * (sigma_3 - np.tanh(alpha * sigma_3) / alpha)
    l2_plus = kappa_2_plus * common_factor
    l2_minus = kappa_2_minus * common_factor

    return l2_plus, l2_minus

In [16]:
D = [20, 500, 750, 1000, 1250, 1500, 1600]
print(f"Kappas are {kappas}")
for k, _ in enumerate(D):
    print(f" ----- Current D is {D[k]} ----- ")
    print("Calculated l is:")
    l1p = l1_plus_formula(kappas[0], D[k], sigma=sigmas[0])
    l2p, l2m = calculate_l2_rates(kappas[2], kappas[3], D[k], D[k], D[k], sigmas[0])
    print(f"l1p:{l1p:.2f}, l2p:{l2p:.2f}, l2m:{l2m:.2f}")

Kappas are [7.91705658e+02 1.50000000e+03 3.43908933e+04 6.61363333e+03
 5.75000000e+00 2.50000000e+01]
 ----- Current D is 20 ----- 
Calculated l is:
l1p:1.54, l2p:29.03, l2m:5.58
 ----- Current D is 500 ----- 
Calculated l is:
l1p:1.65, l2p:123.79, l2m:23.81
 ----- Current D is 750 ----- 
Calculated l is:
l1p:1.65, l2p:129.87, l2m:24.98
 ----- Current D is 1000 ----- 
Calculated l is:
l1p:1.66, l2p:133.15, l2m:25.61
 ----- Current D is 1250 ----- 
Calculated l is:
l1p:1.66, l2p:135.19, l2m:26.00
 ----- Current D is 1500 ----- 
Calculated l is:
l1p:1.66, l2p:136.59, l2m:26.27
 ----- Current D is 1600 ----- 
Calculated l is:
l1p:1.66, l2p:137.04, l2m:26.35


In [17]:
def decoupled_l2_formula(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 8 * np.pi * D * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs

def find_kappa_2(kappa_1_plus, l1_plus_input, D, sigma):
    """
    Defines the self-consistency equation for kappa_1_plus.
    This function will be zero when the correct kappa_1_plus is found.
    """
    # Ensure kappa_1_plus is positive to avoid math errors
    if kappa_1_plus <= 0:
        return np.inf # Return a large number if the guess is non-physical

    lhs = decoupled_l2_formula(kappa_1_plus, D, sigma)
    # Calculate the Right-Hand Side (RHS) of the equation
    rhs = l1_plus_input
    return lhs -rhs


In [18]:
import numpy as np

In [19]:
np.sqrt(716.8818/3000
        )

np.float64(0.48883596430704646)

In [20]:
1.2509e5+2.0848e4

145938.0

In [21]:
np.sqrt(1.2509e5/(200)+2.0848e4/1600)

np.float64(25.268161785139814)